In [1]:
from pathlib import Path
import pandas as pd

from data_ingestion import load_all_raw
from feature_engineering import build_subject_snapshot
from scoring import compute_clean_patient_flags, compute_dqi

# 1. Load all studies and build snapshot
raw_all = load_all_raw()
subject_df = build_subject_snapshot(raw_all)
subject_df = compute_clean_patient_flags(subject_df)
subject_df = compute_dqi(subject_df)

Path("data/processed").mkdir(parents=True, exist_ok=True)
# make every non‑numeric column explicitly string
for col in subject_df.columns:
    if not pd.api.types.is_numeric_dtype(subject_df[col]):
        subject_df[col] = subject_df[col].astype("string")

subject_df.to_parquet("data/processed/subject_site_snapshot.parquet", index=False)



# 2. Site-level aggregation and simple metrics
site_df = (
    subject_df.groupby(["study_id", "site_id"], as_index=False)
    .agg(
        mean_dqi=("dqi", "mean"),
        pct_clean=("clean_patient", "mean"),
        n_subjects=("subject_id", "nunique"),
        n_red=("dqi_band", lambda x: (x == "Red").sum()),
    )
)

site_df.to_parquet("data/processed/site_snapshot.parquet", index=False)



C:\Users\Aritri Baidya\Desktop\MyFiles\Pycharm\Novartis_hackathon\scoring.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  d = d.fillna(0)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier

features = [
    "n_missing_visits", "n_missing_pages", "n_open_queries",
    "n_nonconformant_pages", "n_lab_issues", "n_uncoded_terms",
    "n_open_edrr_issues", "n_sae_pending_actions",
    "pct_crfs_verified", "pct_crfs_signed", "pct_crfs_overdue"
]

X = subject_df[features].fillna(0)
y = subject_df["clean_patient"]

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42,
    class_weight="balanced"
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))


In [ ]:
from data_ingestion import load_all_raw
from feature_engineering import build_subject_snapshot, normalize_keys
from scoring import compute_clean_patient_flags, compute_dqi

raw_all = load_all_raw()

# quick check: what do visits look like?
vis = raw_all["visits"].head()
print(vis.columns)

subject_df = build_subject_snapshot(raw_all)
subject_df = compute_clean_patient_flags(subject_df)
subject_df = compute_dqi(subject_df)
subject_df.head()
from data_ingestion import load_all_raw
raw_all = load_all_raw()

mp = raw_all["missing_pages"].head()
print(mp.columns)


In [ ]:
from data_ingestion import load_all_raw
raw_all = load_all_raw()
vis = raw_all["visits"].head()
print(vis.columns)


In [ ]:
from data_ingestion import load_all_raw
raw_all = load_all_raw()

mp = raw_all["missing_pages"].head()
print(mp.columns)


In [ ]:
from data_ingestion import load_all_raw
raw_all = load_all_raw()

print("MedDRA columns:", raw_all["medra"].columns)
print("WHODD columns:", raw_all["whodd"].columns)


In [ ]:
from data_ingestion import load_all_raw
from feature_engineering import build_subject_snapshot
from scoring import compute_clean_patient_flags, compute_dqi

raw_all = load_all_raw()
for k, df in raw_all.items():
    print(k, "shape:", df.shape, "studies:", df["study_id"].unique() if "study_id" in df.columns else "no study_id")

subject_df = build_subject_snapshot(raw_all)
print("subject_df studies:", subject_df["study_id"].unique())



In [ ]:
import pandas as pd
df = pd.read_parquet("data/processed/subject_site_snapshot.parquet")
print(sorted(df["study_id"].astype(str).unique()))
